# Welcome to Subconscious!

This notebook will walk you through your first API calls with **Subconscious**, a long-horizon reasoning engine that handles high-context workflows.

Let's get you started in 2 simple steps:

1. Paste your API key below
2. Click "Run All" to jump right in

Let's go!

In [ ]:
# You can find your API key on the platform:
# https://www.subconscious.dev/platform

# **Paste your API key here**
api_key = "sk-..."

In [ ]:
# Install the Subconscious Python SDK
# This is the only setup you'll need!

!pip install subconscious-sdk > /dev/null 2>&1

In [ ]:
# Connect to the Subconscious SDK in 2 lines of code.

from subconscious import Subconscious
client = Subconscious(api_key=api_key)

## *You're in.* Let's try it out!

The simplest thing you can do is give the agent an instruction and let it reason on its own — no tools, no extra config.

Key parameters:
- **`engine`** — which model to use (we'll use **`"tim-claude"`**, Subconscious's Claude-backed compound engine)
- **`input.instructions`** — what you want the agent to do
- **`options.await_completion`** — wait for the full answer before returning

In [ ]:
run = client.run(
    engine="tim-claude",
    input={
        "instructions": "Explain what an API is in 3 sentences, as if I'm 10 years old.",
    },
    options={"await_completion": True},
)

print(run.result.answer)

## Using Tools

Agents become much more powerful when you give them **tools**. Subconscious ships with built-in search tools with zero setup.

Here are some popular ones:

| Tool | API Name | What it does |
|------|----------|--------------|
| Fast Search | `fast_search` | Quick factual lookups |
| Web Search | `web_search` | Detailed web research |
| News Search | `news_search` | Search news articles |
| Page Reader | `page_reader` | Read a webpage URL |

To give an agent a tool, pass it along in the `tools` list.

In [ ]:
run = client.run(
    engine="tim-claude",
    input={
        "instructions": "Give me a brief summary of the biggest AI news from this week.",
        "tools": [
            {"type": "platform", "id": "web_search"},
            {"type": "platform", "id": "news_search"},
            # ^^^ multiple tools? no problem -- our engines intelligently decide which to use and when.
        ],
    },
    options={"await_completion": True},
)

print(run.result.answer)

In [ ]:
run = client.run(
    engine="tim-claude",
    input={
        "instructions": "What is React's useEffect hook for? Answer in 2–3 sentences.",
        "tools": [
            {"type": "mcp", "url": "https://docs.gitmcp.io/mcp"},
        ],
    },
    options={"await_completion": True},
)

print(run.result.answer)

## Structured Output with Pydantic

Sometimes you don't want a free-text answer — you want **structured data** you can use in code (JSON with specific fields).

Subconscious supports this via [Pydantic](https://docs.pydantic.dev/) models. You define a schema and the agent's answer will match it.

Here's an example that analyzes the sentiment of a piece of text:

In [ ]:
from pydantic import BaseModel


class SentimentAnalysis(BaseModel):
    sentiment: str        # e.g. "positive", "negative", "neutral"
    confidence: float     # 0.0 to 1.0
    keywords: list[str]   # key words that influenced the sentiment


text_to_analyze = "I absolutely loved the new movie! The acting was superb and the storyline kept me on the edge of my seat."

run = client.run(
    engine="tim-claude",
    input={
        "instructions": f"Analyze the sentiment of this text: '{text_to_analyze}'",
        "answerFormat": SentimentAnalysis,
    },
    options={"await_completion": True},
)

result = run.result.parsed_answer
print(f"Sentiment:  {result['sentiment']}") # We can guarantee that the fields exist and are formatted properly.
print(f"Confidence: {result['confidence']}")
print(f"Keywords:   {result['keywords']}")

## Trying Different Engines

Subconscious offers several engines — each optimized for different trade-offs:

| Engine | API Name | Best for |
|--------|----------|----------|
| **TIM-Claude** | `tim-claude` | **Compound reasoning with Claude** (default in this notebook) |
| TIM | `tim` | General-purpose on Subconscious's unified TIM stack |
| TIM-Edge | `tim-edge` | Speed and efficiency, search-heavy workloads |
| TIMINI | `timini` | Complex reasoning, backed by Gemini-3 Flash |
| TIM-GPT | `tim-gpt` | Complex reasoning, backed by GPT-4.1 |
| TIM-GPT-Heavy | `tim-gpt-heavy` | Maximum capability, backed by GPT-5.2 |

Just change the `engine` parameter to try a different one:

In [ ]:
engines_to_try = ["tim-claude", "tim-gpt-heavy", "tim"]
question = "In one sentence, what causes the Northern Lights?"

for engine in engines_to_try:
    print(f"\n{'=' * 60}")
    print(f"Engine: {engine}")
    print(f"{'=' * 60}")

    run = client.run(
        engine=engine,
        input={
            "instructions": question,
            "tools": [{"type": "platform", "id": "fast_search"}],
        },
        options={"await_completion": True},
    )

    print(run.result.answer)

## What's Next?

You've now covered the core building blocks of Subconscious! Here are some ideas for what to explore next:

- **Custom tools** — connect the agent to your own HTTP endpoints ([docs](https://docs.subconscious.dev/core-concepts/tools))
- **MCP** — add any HTTP MCP server with `{"type": "mcp", "url": "..."}`; details and auth options [here](https://docs.subconscious.dev/core-concepts/tools#mcp-tools)
- **Streaming** — get results in real-time as the agent works ([streaming](https://docs.subconscious.dev/core-concepts/streaming#streaming))
- **Templates** — plug-and-play projects on the [platform](https://www.subconscious.dev/platform/templates)

### Useful Links

| Resource | Link |
|----------|------|
| Documentation | [docs.subconscious.dev](https://docs.subconscious.dev) |
| API Reference | [docs.subconscious.dev/api-reference](https://docs.subconscious.dev/api-reference/introduction) |
| Platform | [subconscious.dev/platform](https://www.subconscious.dev/platform) |

Happy building!